In [ ]:
import pandas as pd
from pathlib import Path
from matplotlib import pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
ano = 2023

In [ ]:
data_dir = Path("data")
INDIR = Path(f"../data/data_processed/{ano}")
OUTDIR_IMG = Path(f"../report/img/{ano}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [ ]:
city_file = INDIR / f"ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_TRATADO_{ano}.csv"
df = pd.read_csv(city_file, sep=",")

In [ ]:
df.head()

In [ ]:
score_columns = ['NATURAL_SCIENCES_SCORE_AVG', 'HUMANITIES_SCORE_AVG', 'LANGUAGES_SCORE_AVG', 'MATH_SCORE_AVG', 'ESSAY_SCORE_AVG']

subject_names = {
    'CN': 'Natural Sciences and its Technologies',
    'MT': 'Mathematics and its Technologies',
    'CH': 'Humanities and its Technologies',
    'LC': 'Languages, Codes and its Technologies',
    'REDACAO': 'Essay'
}

# Paleta Okabe-Ito (color-blind friendly)
mapa_cores = {
    'CN': '#0072B2',      # azul
    'MT': '#E69F00',      # laranja
    'CH': '#009E73',      # verde
    'LC': '#D55E00',      # vermelho-alaranjado
    'REDACAO': '#CC79A7'  # roxo
}

for col in score_columns:
    area_sigla = col.replace('NOTA_', '').replace('_MEDIA', '')
    area_nome = subject_names.get(area_sigla, area_sigla)
    cor_area = mapa_cores.get(area_sigla, '#0072B2')

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=('Histogram', 'Box Plot')
    )

    fig.add_trace(
        go.Histogram(
            x=df[col].round(2),
            nbinsx=200,
            histnorm='probability density',
            marker_color=cor_area,
            name=area_nome
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Box(y=df[col], marker_color=cor_area, name=area_nome),
        row=1, col=2
    )

    fig.update_layout(
        title_text=f'Distribuição de Notas - {area_nome}',
        showlegend=False,
        width=1200,
        height=450,
        template='plotly_white'
    )
    fig.show()

In [ ]:
col_renda = 'FAMILY_INCOME_SM_AVG'
renda_2c = df[col_renda].round(2)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Histogram', 'Box Plot')
)

fig.add_trace(
    go.Histogram(
        x=renda_2c,
        nbinsx=200,
        histnorm='probability density',
        marker_color='#0072B2',
        name='Renda Familiar Média (SM)'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        y=renda_2c,
        marker_color='#0072B2',
        name='Renda Familiar Média (SM)'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Distribuição de Renda Familiar Média (SM)',
    showlegend=False,
    width=1200,
    height=450,
    template='plotly_white'
)

fig.show()

In [ ]:
candidatas_renda = ["FAMILY_INCOME_SM_AVG"]
candidatas_nota = ["OVERALL_SCORE_AVG"]
candidatas_municipio = ["NO_MUNICIPIO", "CITY", "NOME_MUNICIPIO"]
candidatas_uf = ["SG_UF", "STATE", "SIGLA_UF"]

col_municipio = next((c for c in candidatas_municipio if c in df.columns), None)
col_uf = next((c for c in candidatas_uf if c in df.columns), None)

if col_municipio is None or col_uf is None:
    raise KeyError(
        "Colunas de municipio/UF nao encontradas. Disponiveis: "
        f"{list(df.columns)}"
    )

df_plot = df[[
    candidatas_renda[0],
    candidatas_nota[0],
    col_municipio,
    col_uf,
]].dropna()

correlacao = df_plot[candidatas_renda[0]].corr(df_plot[candidatas_nota[0]])

fig = go.Figure(
    data=go.Scatter(
        x=df_plot[candidatas_renda[0]],
        y=df_plot[candidatas_nota[0]],
        mode="markers",
        marker=dict(color="#0072B2", size=6),
        customdata=df_plot[[col_municipio, col_uf]].values,
        hovertemplate=(
            "Nome do Municipio: %{customdata[0]}<br>"
            "UF: %{customdata[1]}<br>"
            "Nota Geral Media: %{y:.2f}<br>"
            "Renda Familiar Media (SM): %{x:.2f}<extra></extra>"
        ),
        name="Municípios"
    )
)

fig.update_layout(
    title=f"Dispersao: Renda Familiar x Nota Geral Média (r = {correlacao:.3f})",
    xaxis_title="Renda Familiar Media (SM)",
    yaxis_title="Nota Geral Media",
    template="plotly_white",
    width=1200,
    height=450,
)

fig.show()